### Structured Output

tructured output allows agents to return data in a specific, predictable format. Instead of parsing natural language responses, you get structured data in the form of JSON objects, Pydantic models, or dataclasses that your application can use directly.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
# os.environ['GEMINI_API_KEY']=os.getenv("GEMINI_API_KEY")
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")
# print("GEMINI_API_KEY:", os.environ['GEMINI_API_KEY'])
os.environ["TAVILY_API_KEY"]=os.getenv("TRAVILY_API_KEY")

from langchain.chat_models import init_chat_model
# model = init_chat_model(model="llama-3.1-8b-instant", model_provider="groq",
#                            api_key=os.getenv("GROQ_API_KEY"))

model = init_chat_model(model="gemini-3-flash-preview", model_provider="google_genai",
                            api_key=os.getenv("GEMINI_API_KEY"))

In [11]:
model.profile

{'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

In [2]:
## Pydantic 

from pydantic import BaseModel, Field


class ResumeMessage(BaseModel):
    name: str = Field(description="Full name of the candidate")
    email: str = Field(description="Email address of the candidate")
    phone: str = Field(description="Phone number of the candidate")
    summary: str = Field(description="Professional summary or objective")
    experience: list[str] = Field(description="List of work experience entries")
    education: list[str] = Field(description="List of education entries")
    skills: list[str] = Field(description="List of professional skills")

In [3]:
## Pydantic message types
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain.agents import create_agent

agent_with_schema = create_agent(
    model=model,
    response_format=ResumeMessage
  
)

# read text from resume.txt
with open("resume.txt", "r", encoding="utf-8") as file:
    resume_text = file.read()
    
messages = [
    SystemMessage(content="You are an expert resume builder. Extract the relevant information from the user's input "),
    HumanMessage(content="Please extract the resume information from the following text:"),
    AIMessage(content="Sure! Please provide the text."), 
    HumanMessage(content=resume_text)
    ]
response = agent_with_schema.invoke({"messages": messages})

In [4]:
response


{'messages': [SystemMessage(content="You are an expert resume builder. Extract the relevant information from the user's input ", additional_kwargs={}, response_metadata={}, id='9e18e55a-df6f-4ff8-810d-82b5b9c0d0b5'),
  HumanMessage(content='Please extract the resume information from the following text:', additional_kwargs={}, response_metadata={}, id='a695aec5-7650-44db-a1e1-95bc5c42f53e'),
  AIMessage(content='Sure! Please provide the text.', additional_kwargs={}, response_metadata={}, id='0e2066a7-55ca-4b8c-af40-58e57ad34565', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='WORK EXPERIENCE Skills\n2021- Current TIAA GBS\nAssociate Specialist\n❑ Leading and mentoring QA team members, promoting skill development\nand collaboration.\n❑ Managing defect triage meetings, analyzing defect trends, and ensures\ntimely resolution of critical issues.\n❑ Worked on classification model for participant contribution change.\n❑ Building tools and utility to be used across QA community

In [10]:
response['structured_response']

ResumeMessage(name='GAUTAM RAWAT', email='gautam.rawat123@gmail.com', phone='+91 – 8939725506', summary='SDET with 10+ Years in Automation Testing, skilled in Artificial Intelligence & Machine Learning.', experience=['2021- Current: Associate Specialist at TIAA GBS - Leading and mentoring QA team, managing defect triage, developing classification models for participant contributions, and building utility tools.', '2020-2021: Lead Engineer Testing at FIS - Microservices testing using Java, RestAssured, and SpringBoot; enhanced CI/CD pipelines; monitored logs with Splunk and performed benchmarking with JMeter.', '2018-2020: Test Analyst at TIAA GBS - Automated manual test cases, managed test cases in JIRA, and prepared status reports.', '2017-2018: Software Test Engineer at Xavient Information Systems - Supported Charter Communications projects, automated manual test cases using Selenium within a Data Driven Framework.', "2015-2017: Software Test Engineer at Cognizant Technology Solution